# 05 — Seattle-Rule Pricing Simulation

Roll the Seattle adjustment rule forward 5 years on synthetic data and watch occupancy and price evolve.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from tests.fixtures import make_synthetic_segments
from phillyparking.demand.synthetic_baseline import baseline_occupancy
from phillyparking.pricing.seattle_rule import adjust_zone_prices
from phillyparking.elasticity.scenarios import apply_elasticity, get_scenario
from phillyparking.revenue.forecast import annual_revenue
segments = make_synthetic_segments(n=200)
if 'price_usd_per_hr' not in segments.columns:
    segments['price_usd_per_hr'] = 2.5
panel = baseline_occupancy(segments)
scenario = get_scenario('central')


## Roll forward 5 years

In [ ]:
history = []
current = segments.copy()
current_panel = panel.copy()
for year in range(1, 6):
    summary_in = current_panel.groupby('zone')['occupancy'].agg(['mean', 'max']).rename(
        columns={'mean': 'avg_occ', 'max': 'peak_occ'})
    new_prices = adjust_zone_prices(current, current_panel)
    rev = annual_revenue(current, current_panel)
    for zone, row in summary_in.iterrows():
        history.append({
            'year': year, 'zone': zone,
            'avg_occ': row['avg_occ'], 'peak_occ': row['peak_occ'],
            'price': float(new_prices.get(zone, np.nan)),
            'revenue': float(rev.get(zone, np.nan)) if hasattr(rev, 'get') else float(rev),
        })
    current['price_usd_per_hr'] = current['zone'].map(new_prices).fillna(current['price_usd_per_hr'])
    current_panel = apply_elasticity(current_panel, current, scenario)
hist = pd.DataFrame(history)
hist


## Price trajectories per zone

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for zone, sub in hist.groupby('zone'):
    ax.plot(sub['year'], sub['price'], marker='o', label=zone)
ax.set_xlabel('year'); ax.set_ylabel('price ($/hr)')
ax.set_title('Seattle-rule price trajectory by zone')
ax.legend(); plt.show()


## Next steps

- Aggregate revenue across scenarios in `06_revenue_scenarios.ipynb`.
